### Overview

This notebook implements a lightweight rule-based language filtering pipeline
for conversational tree-structured data (e.g., ShareGPT-style datasets).

Goal:
- Load raw conversation trees
- Detect English prompts with deterministic heuristics
- Filter non-English trees
- Save the cleaned dataset
- Inspect random sample prompts

This version separates:
- configuration
- processing logic
- execution
- mini tests

In [ ]:
import re
import json
import random
from pathlib import Path
import unicodedata
from time import perf_counter


# ----------------------------
# CONFIG
# ----------------------------

def find_project_root(start=None, marker="01_data"):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root containing '{marker}'")

PROJECT_ROOT = find_project_root()


CONFIG = {
    "data_dir_raw": PROJECT_ROOT / "01_data" / "01_raw" / "messy_dataset",
    "data_dir_processed": PROJECT_ROOT / "01_data" / "02_processed",
    "input_files": [
        "sg_90k_part1.json",
        "sg_90k_part2.json",
    ],
    "output_file": "sharegpt_english.json",
    "min_words": 5,
    "min_english_stopwords": 2,
    "min_margin": 2,
    "sample_size": 10,
    "sample_max_chars": 700,
    "random_seed": 42,
    "min_latin_ratio": 0.8
}

In [ ]:
# ----------------------------
# STOPWORDS
# ----------------------------

WORD_RE = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ']+")

STOPWORDS = {
    "en": {
        "the","and","to","of","a","in","that","is","it","for","on","with","as","this",
        "be","are","you","your","can","will","not","or","if","we","from","by","at","an",
        "have","has","do","does","what","which","when","where","how","why","about","would",
        "could","should","there","their","they","them","these","those","one","more","use",
        "using","make","write","explain","please","help"
    },
    "de": {
        "der","die","das","und","ist","ich","du","sie","er","es","wir","nicht","ein","eine",
        "zu","mit","auf","für","von","wie","was","warum","wenn","dass","kann","sind","haben",
        "werden","auch","oder","aber","bitte","über","hallo"
    },
    "nl": {
        "de","het","een","en","van","ik","je","niet","dat","dit","die","op","voor","met",
        "als","zijn","hebben","worden","wat","waar","waarom","hoe"
    },
    "vi": {
        "và","là","của","có","cho","trong","một","không","tôi","bạn","hãy","này","để"
    },
    "es": {
        "el","la","los","las","y","de","que","en","un","una","es","por","para","con","como","no","se"
    },
    "fr": {
        "le","la","les","et","de","des","du","un","une","est","dans","pour","avec","que","qui"
    },
    "pt": {
        "o","a","os","as","e","de","que","em","um","uma","é","para","com","como","não"
    },
    "it": {
        "il","lo","la","gli","le","e","di","che","in","un","una","è","per","con"
    },
}

In [25]:
def get_first_user_prompt(tree):
    conversations = tree.get("conversations", [])

    for message in conversations:
        if message.get("from") == "human":
            return message.get("value", "")

    return ""


def normalize_words(text):
    return [w.lower().strip("'") for w in WORD_RE.findall(text)]


def count_stopwords(words, stopwords_dict):
    return {
        lang: sum(w in stopwords for w in words)
        for lang, stopwords in stopwords_dict.items()
    }


def get_language_scores_from_prompt(prompt, stopwords_dict):
    words = normalize_words(prompt)
    stopword_scores = count_stopwords(words, stopwords_dict)

    english_score = stopword_scores["en"]
    strongest_other = max(v for k, v in stopword_scores.items() if k != "en")

    return {
        "words": words,
        "word_count": len(words),
        "stopword_scores": stopword_scores,
        "english_score": english_score,
        "strongest_other": strongest_other,
    }


def latin_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return 0.0
    latin = 0
    for ch in letters:
        try:
            if "LATIN" in unicodedata.name(ch):
                latin += 1
        except ValueError:
            pass
    return latin / len(letters)


def is_english_tree(tree, config, stopwords_dict):
    prompt = get_first_user_prompt(tree)
    scores = get_language_scores_from_prompt(prompt, stopwords_dict)

    english_score = scores["english_score"]
    strongest_other = scores["strongest_other"]
    margin = english_score - strongest_other

    return (
        scores["word_count"] >= config["min_words"]
        and english_score >= config["min_english_stopwords"]
        and margin >= config["min_margin"]
        and latin_ratio(prompt) >= config["min_latin_ratio"]
    )


def filter_english_trees(data, config, stopwords_dict):
    return [
        tree for tree in data
        if is_english_tree(tree, config, stopwords_dict)
    ]

In [26]:
def load_json(path):
    try:
        with Path(path).open("r", encoding="utf-8") as file:
            return json.load(file)
    except FileNotFoundError:
        print(f"File not found: {path}")
        raise


def load_all_json(paths):
    data = []
    for path in paths:
        data.extend(load_json(path))
    return data


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)

In [27]:
def show_random_questions(data, n=10, max_chars=700, seed="random_seed"):
    if seed is not None:
        random.seed(seed)

    sample_size = min(n, len(data))

    for i in random.sample(range(len(data)), sample_size):
        tree = data[i]
        prompt = get_first_user_prompt(tree)

        print("=" * 100)
        print(f"Index: {i}")
        print(prompt[:max_chars])
        

def show_borderline_cases(data, config, stopwords_dict, n=10, max_chars=250):
    borderline = []

    for i, tree in enumerate(data):
        prompt = get_first_user_prompt(tree)
        scores = get_language_scores_from_prompt(prompt, stopwords_dict)

        english_score = scores["english_score"]
        strongest_other = scores["strongest_other"]
        margin = english_score - strongest_other
        word_count = scores["word_count"]

        is_english = (
            word_count >= config["min_words"]
            and english_score >= config["min_english_stopwords"]
            and margin >= config["min_margin"]
        )

        if is_english:
            borderline.append({
                "index": i,
                "prompt": prompt,
                "word_count": word_count,
                "english_score": english_score,
                "strongest_other": strongest_other,
                "margin": margin,
                "stopword_scores": scores["stopword_scores"],
            })

    borderline = sorted(
        borderline,
        key=lambda x: (x["margin"], x["english_score"], x["word_count"])
    )

    for case in borderline[:n]:
        print("=" * 100)
        print(f"Index: {case['index']}")
        print(f"Words: {case['word_count']}")
        print(f"EN score: {case['english_score']}")
        print(f"Strongest other: {case['strongest_other']}")
        print(f"Margin: {case['margin']}")
        print(f"Scores: {case['stopword_scores']}")
        print(case["prompt"][:max_chars])

In [ ]:
# ----------------------------
# MINI TESTS
# ----------------------------

example_en = {
    "conversations": [
        {"from": "human", "value": "How can I improve my Python code for data analysis?"},
        {"from": "assistant", "value": "You can start by..."}
    ]
}

example_de = {
    "conversations": [
        {"from": "human", "value": "Wie kann ich meinen Python Code verbessern?"},
        {"from": "assistant", "value": "Du kannst..."}
    ]
}

example_short = {
    "conversations": [
        {"from": "human", "value": "Hello there"},
    ]
}

assert get_first_user_prompt(example_en) == "How can I improve my Python code for data analysis?"
assert normalize_words("Hello, World!") == ["hello", "world"]

scores = count_stopwords(["how", "can", "i", "improve", "my", "code"], STOPWORDS)
assert scores["en"] >= 2

assert is_english_tree(example_en, CONFIG, STOPWORDS) is True
assert is_english_tree(example_de, CONFIG, STOPWORDS) is False
assert is_english_tree(example_short, CONFIG, STOPWORDS) is False

print("All mini-tests passed.")

All mini-tests passed.


In [29]:
input_paths = [
    CONFIG["data_dir_raw"] / filename
    for filename in CONFIG["input_files"]
]

output_path = CONFIG["data_dir_processed"] / CONFIG["output_file"]

print("Input paths:")
for path in input_paths:
    print("-", path)

print("\nOutput path:")
print("-", output_path)

Input paths:
- c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\01_raw\messy_dataset\sg_90k_part1.json
- c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\01_raw\messy_dataset\sg_90k_part2.json

Output path:
- c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\02_processed\sharegpt_english.json


In [ ]:
# ----------------------------
# LOAD / SAVE DATA
# ----------------------------

start = perf_counter()

data = load_all_json(input_paths)
english_data = filter_english_trees(data, CONFIG, STOPWORDS)

duration = perf_counter() - start

print(f"Original: {len(data)}")
print(f"English only: {len(english_data)}")
print(f"Removed: {len(data) - len(english_data)}")
print(f"Time: {duration:.2f}s")
print(f"Trees/sec: {len(data)/duration:.0f}")

Original: 90665
English only: 56024
Removed: 34641
Time: 19.68s
Trees/sec: 4606


In [ ]:
# ----------------------------
# RANDOM QUESTIONS
# ----------------------------

show_random_questions(
    english_data,
    n=CONFIG["sample_size"],
    max_chars=CONFIG["sample_max_chars"],
    seed=CONFIG["random_seed"],
)

Index: 41905
Please write me a Lisp Flavored Erlang (LFE) function which converts Fahrenheit to Celsius
Index: 7296
act as an expert pharmacologist. explain to a physician the potential interactions of the medications named here:

 Wellbutrin XL to 150 mg daily for attention and concentration. 
Continue propranolol 20 mg BID for tremor and anxiety. D/c or switch to another medication if any evidence of asthma exacerbation.
Continue Abilify 20 mg daily for mood stability
Continue Lamictal 100 mg PO QD for mood stability 
Continue trazodone 150 mg PO QHS for sleep 
Pt takes OTC melatonin 10 mg nightly for sleep 
Patient is prescribed Cymbalta 40 mg BID for neuralgia by her PCP
Pt also taking iron, biotin, D3, and B12"
Index: 1639
i got a job offer and here is the offer "Hi,

 

As discussed between Gerard and you I am happy that as a result of our pleasant talks we can get this job-offer for the Medior Dev-Ops role over to you.

 

- Contract; first contract term is 1 year (first month i

In [ ]:
# ----------------------------
# DIAGNOSTICS
# ----------------------------

def diagnose_tree(tree, config, stopwords_dict):
    prompt = get_first_user_prompt(tree)
    scores = get_language_scores_from_prompt(prompt, stopwords_dict)

    english_score = scores["english_score"]
    strongest_other = scores["strongest_other"]
    margin = english_score - strongest_other

    is_english = (
        scores["word_count"] >= config["min_words"]
        and english_score >= config["min_english_stopwords"]
        and margin >= config["min_margin"]
    )

    return {
        "prompt": prompt,
        "word_count": scores["word_count"],
        "english_score": english_score,
        "strongest_other": strongest_other,
        "margin": margin,
        "is_english": is_english,
    }

for tree in data[:10]:
    result = diagnose_tree(tree, CONFIG, STOPWORDS)
    print(result)

{'prompt': "root@openvpn:/home/openvpn# ./openvpn-install.sh\nWelcome to OpenVPN-install!\nThe git repository is available at: https://github.com/angristan/openvpn-install\n\nIt looks like OpenVPN is already installed.\n\nWhat do you want to do?\n   1) Add a new user\n   2) Revoke existing user\n   3) Remove OpenVPN\n   4) Exit\nSelect an option [1-4]: 1\n\nTell me a name for the client.\nThe name must consist of alphanumeric character. It may also include an underscore or a dash.\nClient name: naam\n\nDo you want to protect the configuration file with a password?\n(e.g. encrypt the private key with a password)\n   1) Add a passwordless client\n   2) Use a password for the client\nSelect an option [1-2]: 1\n\nNote: using Easy-RSA configuration from: /etc/openvpn/easy-rsa/vars\nUsing SSL: openssl OpenSSL 3.0.2 15 Mar 2022 (Library: OpenSSL 3.0.2 15 Mar 2022)\n-----\nUsing configuration from /etc/openvpn/easy-rsa/pki/easy-rsa-54848.BT2FXv/tmp.dFLd6V\nEnter pass phrase for /etc/openvpn/ea

In [ ]:
# ----------------------------
# BORDERLINE CASES
# ----------------------------

show_borderline_cases(
    english_data,
    config=CONFIG,
    stopwords_dict=STOPWORDS,
    n=30,
    max_chars=300,
)

Index: 50
Words: 5
EN score: 2
Strongest other: 0
Margin: 2
Scores: {'en': 2, 'de': 0, 'nl': 0, 'vi': 0, 'es': 0, 'fr': 0, 'pt': 0, 'it': 0}
how many kg for 2-3 pounds?
Index: 91
Words: 5
EN score: 2
Strongest other: 0
Margin: 2
Scores: {'en': 2, 'de': 0, 'nl': 0, 'vi': 0, 'es': 0, 'fr': 0, 'pt': 0, 'it': 0}
How to load image here ?
Index: 431
Words: 5
EN score: 2
Strongest other: 0
Margin: 2
Scores: {'en': 2, 'de': 0, 'nl': 0, 'vi': 0, 'es': 0, 'fr': 0, 'pt': 0, 'it': 0}
what are game developers KPIs?
Index: 550
Words: 5
EN score: 2
Strongest other: 0
Margin: 2
Scores: {'en': 2, 'de': 0, 'nl': 0, 'vi': 0, 'es': 0, 'fr': 0, 'pt': 0, 'it': 0}
rewrite the ending to GOT
Index: 913
Words: 5
EN score: 2
Strongest other: 0
Margin: 2
Scores: {'en': 2, 'de': 0, 'nl': 0, 'vi': 0, 'es': 0, 'fr': 0, 'pt': 0, 'it': 0}
2 / 2how to run airflow locally
Index: 1024
Words: 5
EN score: 2
Strongest other: 0
Margin: 2
Scores: {'en': 2, 'de': 0, 'nl': 0, 'vi': 0, 'es': 0, 'fr': 0, 'pt': 0, 'it': 0}
hi do y

In [ ]:
# ----------------------------
# CHECK CHANGES IN CONFIGS
# ----------------------------

configs_to_compare = [
    {
        "name": "baseline",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 1,
        "min_latin_ratio": 0.0,
    },
    {
        "name": "higher_margin",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 2,
        "min_latin_ratio": 0.0,
    },
    {
        "name": "more_english_evidence",
        "min_words": 5,
        "min_english_stopwords": 3,
        "min_margin": 1,
        "min_latin_ratio": 0.0,
    },
    {
        "name": "latin_ratio_check",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 1,
        "min_latin_ratio": 0.8,
    },
]

for cfg in configs_to_compare:
    filtered = filter_english_trees(data, cfg, STOPWORDS)
    print("=" * 80)
    print(cfg["name"])
    print(f"English only: {len(filtered)}")
    print(f"Removed: {len(data) - len(filtered)}")

baseline
English only: 58921
Removed: 31744
higher_margin
English only: 57276
Removed: 33389
more_english_evidence
English only: 55681
Removed: 34984
latin_ratio_check
English only: 57613
Removed: 33052


In [35]:
save_json(english_data, output_path)